# 🔄 Multimodal Preprocessing - HDF5 Format (Train/Dev/Test)

**Extract PhoBERT + ResNet features and save as optimized HDF5**

📥 Input: `../data/json/news_data_vifactcheck_*_labeled.json`
📤 Output: `../processed_data/hdf5/`

**Features:**

-   **Text**: PhoBERT (vinai/phobert-base) - token embeddings
-   **Images**: ResNet50 - feature extraction
-   **Captions**: Cleaned captions from preprocessing
-   **Format**: HDF5 with gzip compression

**Splits**: Train, Dev, Test processed separately


In [1]:
# Install dependencies
import subprocess
import sys

packages = [
    "h5py",
    "transformers",
    "torch",
    "torchvision",
    "Pillow",
    "tqdm",
    "numpy",
    "pandas",
]

for pkg in packages:
    try:
        __import__(pkg.replace("-", "_").lower())
        print(f"✅ {pkg}")
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
        print(f"📦 Installed {pkg}")

✅ h5py


/opt/miniconda3/envs/fake_news/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ transformers
✅ torch
✅ torchvision
📦 Installed Pillow
✅ tqdm
✅ numpy
✅ pandas


In [2]:
# Setup
import sys
import os
import json
import h5py
import numpy as np
import torch
import gc
from pathlib import Path
from tqdm import tqdm
from PIL import Image

# Transformers
from transformers import AutoTokenizer, AutoModel

# Torch
from torchvision import models, transforms

project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / "src"))

print(f"✅ Ready: {project_root}")

✅ Ready: /Users/haila/Library/CloudStorage/GoogleDrive-latruonghai@gmail.com/My Drive/Thesis_Final/fake-new-detection/notebooks


In [3]:
# Configuration
CONFIG = {
    "text_model": "vinai/phobert-base",
    "image_model": "resnet50",
    "max_length": 512,
    "batch_size": 32,
    "text_feature_dim": 768,
    "image_feature_dim": 2048,
    "input_dir": "../data/json",
    "output_dir": "../processed_data/hdf5",
    "placeholder_dir": "../placeholder_images",
    "splits": ["train", "dev", "test"],
}

os.makedirs(CONFIG["output_dir"], exist_ok=True)
os.makedirs(CONFIG["placeholder_dir"], exist_ok=True)

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)
print(f"🔧 Device: {device}")
print(f"📦 Batch size: {CONFIG['batch_size']}")

🔧 Device: mps
📦 Batch size: 32


In [4]:
# Initialize PhoBERT
print("🔤 Loading PhoBERT...")
tokenizer = AutoTokenizer.from_pretrained(CONFIG["text_model"])
text_model = AutoModel.from_pretrained(CONFIG["text_model"]).to(device)
text_model.eval()
print("✅ PhoBERT ready")

🔤 Loading PhoBERT...
✅ PhoBERT ready


In [5]:
# Initialize ResNet
print("🖼️ Loading ResNet50...")
image_model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
image_model = torch.nn.Sequential(*list(image_model.children())[:-1])
image_model = image_model.to(device)
image_model.eval()

image_transform = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)
print("✅ ResNet50 ready")

🖼️ Loading ResNet50...
✅ ResNet50 ready


In [6]:
# Feature extraction functions
def extract_text_features(texts_batch):
    """Extract PhoBERT embeddings for batch of texts."""
    inputs = tokenizer(
        texts_batch,
        return_tensors="pt",
        max_length=CONFIG["max_length"],
        padding="max_length",
        truncation=True,
    ).to(device)

    with torch.no_grad():
        outputs = text_model(**inputs)
        embeddings = outputs.last_hidden_state.cpu().numpy()

    return embeddings


def extract_image_features(image_paths_batch):
    """Extract ResNet features for batch of images."""
    images = []
    for path in image_paths_batch:
        try:
            img = Image.open(path).convert("RGB")
            img_tensor = image_transform(img)
            images.append(img_tensor)
        except:
            placeholder = torch.randn(3, 224, 224)
            images.append(placeholder)

    images_tensor = torch.stack(images).to(device)

    with torch.no_grad():
        features = image_model(images_tensor)
        features = features.squeeze(-1).squeeze(-1).cpu().numpy()

    return features

In [ ]:
# Data loading function
def load_split_data(split_name):
    """Load labeled data for a specific split (train/dev/test)."""
    data_dir = Path(CONFIG["input_dir"])
    # Prefer _labeled.json (with recovered ViFactCheck labels)
    json_file = data_dir / f"news_data_vifactcheck_{split_name}_labeled.json"
    if not json_file.exists():
        # Fallback to _cleaned.json
        json_file = data_dir / f"news_data_vifactcheck_{split_name}_cleaned.json"
        print(f"⚠️  _labeled.json not found, falling back to: {json_file.name}")

    if not json_file.exists():
        print(f"❌ File not found: {json_file}")
        return None

    with open(json_file, "r", encoding="utf-8") as f:
        articles = json.load(f)

    print(f"✅ Loaded {len(articles)} articles from {json_file.name}")

    texts = []
    image_paths = []
    captions = []
    labels = []
    metadata = []

    for idx, article in enumerate(articles):
        title = article.get("title", "")
        content = article.get("content", "") or article.get("text", "")
        text = f"{title}. {content}".strip() if title else content

        if not text.strip():
            continue

        images = article.get("images", [])
        img_path = None
        caption = ""

        if images and len(images) > 0:
            first_img = images[0]
            if isinstance(first_img, dict):
                img_path = first_img.get("folder_path", "")
                caption = first_img.get("caption", "")
            elif isinstance(first_img, str):
                img_path = first_img

        label = article.get("label", 0)
        if isinstance(label, str):
            label = 1 if label.lower() in ["fake", "false", "1"] else 0

        texts.append(text.strip())
        image_paths.append(img_path)
        captions.append(caption)
        labels.append(int(label))
        metadata.append(
            {
                "idx": idx,
                "title": title,
                "source_url": article.get("source_url", ""),
                "caption": caption,
            }
        )

    # Report label distribution
    from collections import Counter

    label_dist = Counter(labels)
    print(f"🏷️ Labels: {len(label_dist)} classes — {dict(label_dist)}")

    return {
        "texts": texts,
        "image_paths": image_paths,
        "captions": captions,
        "labels": labels,
        "metadata": metadata,
    }

In [8]:
# Create placeholder images
def create_placeholder_image(idx):
    """Create placeholder image for missing paths."""
    placeholder_dir = Path(CONFIG["placeholder_dir"])
    placeholder_path = placeholder_dir / f"placeholder_{idx}.jpg"

    if not placeholder_path.exists():
        placeholder_array = np.random.randint(128, 200, (224, 224, 3), dtype=np.uint8)
        Image.fromarray(placeholder_array).save(placeholder_path)

    return str(placeholder_path)


def process_image_paths(image_paths):
    """Process image paths, creating placeholders where needed."""
    processed = []
    for i, img_path in enumerate(image_paths):
        full_path = project_root / img_path if img_path else None
        if not full_path or not full_path.exists():
            processed.append(create_placeholder_image(i))
        else:
            processed.append(str(full_path))
    return processed

In [9]:
# Batch processing function
def process_split(split_name, data):
    """Process one data split and save to HDF5."""
    print(f"\n{'='*50}")
    print(f"Processing {split_name.upper()} split")
    print(f"{'='*50}")

    texts = data["texts"]
    image_paths = data["image_paths"]
    captions = data["captions"]
    labels = data["labels"]
    metadata = data["metadata"]

    total_samples = len(texts)
    print(f"📊 Samples: {total_samples}")

    print("🖼️ Processing image paths...")
    processed_paths = process_image_paths(image_paths)

    BATCH_SIZE = CONFIG["batch_size"]
    num_batches = (total_samples + BATCH_SIZE - 1) // BATCH_SIZE
    print(f"🔢 Batches: {num_batches}")

    all_text_features = []
    all_image_features = []

    for batch_idx in tqdm(range(num_batches), desc=f"Processing {split_name}"):
        start_idx = batch_idx * BATCH_SIZE
        end_idx = min((batch_idx + 1) * BATCH_SIZE, total_samples)

        batch_texts = texts[start_idx:end_idx]
        batch_image_paths = processed_paths[start_idx:end_idx]

        text_feats = extract_text_features(batch_texts)
        image_feats = extract_image_features(batch_image_paths)

        all_text_features.append(text_feats)
        all_image_features.append(image_feats)

        del text_feats, image_feats
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        elif torch.backends.mps.is_available():
            torch.mps.empty_cache()

    print("\n🔄 Concatenating features...")
    text_features_all = np.vstack(all_text_features)
    image_features_all = np.vstack(all_image_features)
    labels_all = np.array(labels)

    print(f"✅ Text features: {text_features_all.shape}")
    print(f"✅ Image features: {image_features_all.shape}")

    output_file = Path(CONFIG["output_dir"]) / f"vifactcheck_{split_name}.h5"
    print(f"\n💾 Saving to: {output_file}")

    with h5py.File(output_file, "w") as f:
        f.create_dataset(
            "text_features",
            data=text_features_all.astype(np.float32),
            compression="gzip",
            compression_opts=4,
        )
        f.create_dataset(
            "image_features",
            data=image_features_all.astype(np.float32),
            compression="gzip",
            compression_opts=4,
        )
        f.create_dataset(
            "labels",
            data=labels_all.astype(np.int32),
            compression="gzip",
        )
        captions_array = np.array(captions, dtype=h5py.string_dtype())
        f.create_dataset(
            "captions",
            data=captions_array,
            compression="gzip",
        )
        metadata_json = json.dumps(metadata)
        f.create_dataset(
            "metadata",
            data=metadata_json,
            dtype=h5py.string_dtype(),
        )

    file_size = output_file.stat().st_size / 1024 / 1024
    print(f"✅ Saved: {file_size:.1f} MB")
    return output_file

In [10]:
# Process all splits
results = {}

for split in CONFIG["splits"]:
    data = load_split_data(split)
    if data:
        output_file = process_split(split, data)
        results[split] = output_file
    else:
        print(f"⚠️ Skipping {split} - no data found")

print(f"\n{'='*50}")
print("✅ All splits processed!")
print(f"{'='*50}")
for split, path in results.items():
    size_mb = path.stat().st_size / 1024 / 1024
    print(f"  {split}: {path.name} ({size_mb:.1f} MB)")

✅ Loaded 4509 articles from news_data_vifactcheck_train_cleaned.json

Processing TRAIN split
📊 Samples: 3376
🖼️ Processing image paths...
🔢 Batches: 106


Processing train: 100%|██████████| 106/106 [53:41<00:00, 30.39s/it]



🔄 Concatenating features...
✅ Text features: (3376, 512, 768)
✅ Image features: (3376, 2048)

💾 Saving to: ../processed_data/hdf5/vifactcheck_train.h5
✅ Saved: 3624.2 MB
✅ Loaded 634 articles from news_data_vifactcheck_dev_cleaned.json

Processing DEV split
📊 Samples: 465
🖼️ Processing image paths...
🔢 Batches: 15


Processing dev: 100%|██████████| 15/15 [00:39<00:00,  2.64s/it]



🔄 Concatenating features...
✅ Text features: (465, 512, 768)
✅ Image features: (465, 2048)

💾 Saving to: ../processed_data/hdf5/vifactcheck_dev.h5
✅ Saved: 515.5 MB
✅ Loaded 1282 articles from news_data_vifactcheck_test_cleaned.json

Processing TEST split
📊 Samples: 973
🖼️ Processing image paths...
🔢 Batches: 31


Processing test: 100%|██████████| 31/31 [01:05<00:00,  2.10s/it]



🔄 Concatenating features...
✅ Text features: (973, 512, 768)
✅ Image features: (973, 2048)

💾 Saving to: ../processed_data/hdf5/vifactcheck_test.h5
✅ Saved: 1045.0 MB

✅ All splits processed!
  train: vifactcheck_train.h5 (3624.2 MB)
  dev: vifactcheck_dev.h5 (515.5 MB)
  test: vifactcheck_test.h5 (1045.0 MB)


In [11]:
# Validation
print("\n🔍 Validating HDF5 files...")

for split, h5_path in results.items():
    print(f"\n{split.upper()}:")
    with h5py.File(h5_path, "r") as f:
        for key in f.keys():
            dataset = f[key]
            print(f"  {key}: {dataset.shape}, {dataset.dtype}")
        print(f"  Sample captions: {f['captions'][:3]}")


🔍 Validating HDF5 files...

TRAIN:
  captions: (3376,), object
  image_features: (3376, 2048), float32
  labels: (3376,), int32
  metadata: (), object
  text_features: (3376, 512, 768), float32
  Sample captions: [b'Ph\xc3\xb3 Th\xe1\xbb\xa7 t\xc6\xb0\xe1\xbb\x9bng Tr\xe1\xba\xa7n H\xe1\xbb\x93ng H\xc3\xa0: C\xc3\xa1c t\xc3\xa1c ph\xe1\xba\xa9m truy\xe1\xbb\x81n h\xc3\xacnh \xc4\x91\xc3\xa3 vun \xc4\x91\xe1\xba\xafp, l\xc3\xa0m gi\xc3\xa0u cho n\xe1\xbb\x81n v\xc4\x83n h\xc3\xb3a Vi\xe1\xbb\x87t Nam ti\xc3\xaan ti\xe1\xba\xbfn, \xc4\x91\xe1\xba\xadm \xc4\x91\xc3\xa0 b\xe1\xba\xa3n s\xe1\xba\xafc d\xc3\xa2n t\xe1\xbb\x99c, g\xc3\xb3p ph\xe1\xba\xa7n t\xe1\xba\xa1o d\xe1\xbb\xb1ng m\xc3\xb4i tr\xc6\xb0\xe1\xbb\x9dng v\xc4\x83n h\xc3\xb3a l\xc3\xa0nh m\xe1\xba\xa1nh v\xc3\xa0 x\xc3\xa2y d\xe1\xbb\xb1ng con ng\xc6\xb0\xe1\xbb\x9di Vi\xe1\xbb\x87t Nam nh\xc3\xa2n c\xc3\xa1ch, tr\xc3\xa1ch nhi\xe1\xbb\x87m, h\xe1\xbb\x99i nh\xe1\xba\xadp'
 b'Trung t\xc3\xa2m Anh ng\xe1\xbb\xaf ILA'
 b'Quan 